In [ ]:
import sys
import glob
import pickle
import numpy as np
import pandas as pd
from functools import reduce
import matplotlib.pyplot as plt

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")

import Chronocell
from simulate_data import *

In [ ]:
prefix = "janssens_2025_preprint_Chronocell_eLNPs_var>1.5"

In [ ]:
# Genes in Chronocell object (column order)
chronocell_genes = pd.read_csv(f"data/{prefix}_traj_X_colnames.csv") 

 # Subset to genes with  protein measurements
mapped_identifiers = pd.read_csv("data/janssens_2025_preprint_author_counts_eLNP_RNA_vs_ADT_corrs.csv")
mask = chronocell_genes['Gene_symbol'].isin(mapped_identifiers['Gene']).tolist()

# Subset Chronocell genes to those in X_fwd object
traj_genes = chronocell_genes.iloc[mask]

# Get traj object from running Chronocell
import pickle
with open(f"data/{prefix}_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)
    
# Subset traj object to genes of interest

Y = traj.X
Q = traj.Q[:, 0, :] 
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo

Y = Y[:, mask, :]
theta = theta[mask, :]
theta_ = theta.copy()
a0 = theta_[:, 0] # Starting RNA abundance 
a = theta_[:, 1:len(topo.flatten())] 
beta = theta_[:, -2] # Splicing rate
alpha = a * beta[:, None] # Transcription rates are normalized by splicing rate; removing this factor now
gamma = theta_[:, -1] # Degradation rate

In [ ]:
# Get X_fwd for each gene (order matches traj_genes order)

import pickle
with open(f"data/{prefix}_X_fwd_per_gene.pkl", "rb") as f:
    X_fwd_per_gene = pickle.load(f)

import pickle
with open(f"data/{prefix}_states_per_gene.pkl", "rb") as f:
    states_per_gene = pickle.load(f)

## Half-life data

In [ ]:
data_dir = "/mnt/lareaulab/reliscu/projects/Chronocell/data/experimental_parameters/protein_degradation_rates"
file_list = glob.glob(f"{data_dir}/*standardized.csv")

# Don't include brain samples because half lives are super long / doesn't seem representative
exclude = ["Cerebellum", "Cortex"]
file_list = [f for f in file_list if not any(term in f for term in exclude)]

subset_columns = ['Gene', 'Half-life']

df_list = []
for file in [file for file in file_list if "Mouse" in file]:
    df = pd.read_csv(file)
    study = df['Study'].iloc[0]
    cell_type = df['Cell_type'].iloc[0]
    df = df[subset_columns]
    df = df.rename(columns={"Half-life": f"Half-life_{study}_{cell_type}", 
                            "Cell_type": f"Cell_type_{study}"}) 
    df_list.append(df)